In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Ketaksamaan CHSH

*Anggaran penggunaan: Dua minit pada pemproses Heron r2 (NOTA: Ini adalah anggaran sahaja. Masa jalan sebenar mungkin berbeza.)*
## Latar belakang
Dalam tutorial ini, kamu akan menjalankan eksperimen pada komputer kuantum untuk menunjukkan pelanggaran ketaksamaan CHSH menggunakan primitif Estimator.

Ketaksamaan CHSH, dinamakan sempena pengarang-pengarangnya Clauser, Horne, Shimony, dan Holt, digunakan untuk membuktikan teorem Bell secara eksperimental (1969). Teorem ini menyatakan bahawa teori pembolehubah tersembunyi setempat tidak dapat menjelaskan beberapa kesan keterjaitan dalam mekanik kuantum. Pelanggaran ketaksamaan CHSH digunakan untuk menunjukkan bahawa mekanik kuantum tidak serasi dengan teori pembolehubah tersembunyi setempat. Ini adalah eksperimen penting untuk memahami asas mekanik kuantum.

Hadiah Nobel Fizik 2022 telah dianugerahkan kepada Alain Aspect, John Clauser dan Anton Zeilinger sebahagiannya atas kerja perintis mereka dalam sains maklumat kuantum, khususnya eksperimen mereka dengan foton yang terjalin yang menunjukkan pelanggaran ketaksamaan Bell.
## Keperluan
Sebelum memulakan tutorial ini, pastikan kamu telah memasang yang berikut:

* Qiskit SDK v1.0 atau lebih baru, dengan sokongan [visualisasi](https://docs.quantum.ibm.com/api/qiskit/visualization)
* Qiskit Runtime (`pip install qiskit-ibm-runtime`) v0.22 atau lebih baru
## Persediaan

In [1]:
# General
import numpy as np

# Qiskit imports
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Qiskit Runtime imports
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

# Plotting routines
import matplotlib.pyplot as plt
import matplotlib.ticker as tck

## Langkah 1: Petakan input klasik kepada masalah kuantum
Untuk eksperimen ini, kita akan mencipta pasangan terjalin di mana kita mengukur setiap Qubit pada dua asas yang berbeza. Kita akan melabelkan asas untuk Qubit pertama sebagai $A$ dan $a$ serta asas untuk Qubit kedua sebagai $B$ dan $b$. Ini membolehkan kita mengira kuantiti CHSH $S_1$:

$$
S_1 = A(B-b) + a(B+b).
$$

Setiap nilai boleh jadi $+1$ atau $-1$. Jelas sekali, salah satu daripada sebutan $B\pm b$ mesti $0$, dan yang satu lagi mesti $\pm 2$. Oleh itu, $S_1 = \pm 2$. Nilai purata $S_1$ mesti memenuhi ketaksamaan:

$$
|\langle S_1 \rangle|\leq 2.
$$

Mengembangkan $S_1$ dalam sebutan $A$, $a$, $B$, dan $b$ menghasilkan:

$$
|\langle S_1 \rangle| = |\langle AB \rangle - \langle Ab \rangle + \langle aB \rangle + \langle ab \rangle| \leq 2
$$

Kamu boleh mentakrifkan kuantiti CHSH lain $S_2$:

$$
S_2 = A(B+b) - a(B-b),
$$

Ini membawa kepada ketaksamaan lain:

$$
|\langle S_2 \rangle| = |\langle AB \rangle + \langle Ab \rangle - \langle aB \rangle + \langle ab \rangle| \leq 2
$$

Jika mekanik kuantum boleh diterangkan oleh teori pembolehubah tersembunyi setempat, ketaksamaan sebelumnya mesti benar. Namun, seperti yang ditunjukkan dalam tutorial ini, ketaksamaan-ketaksamaan ini boleh dilanggar dalam komputer kuantum. Oleh itu, mekanik kuantum tidak serasi dengan teori pembolehubah tersembunyi setempat.

Kalau kamu nak belajar lebih banyak teori, terokai [Keterjaitan dalam Tindakan](/learning/courses/basics-of-quantum-information/entanglement-in-action/chsh-game) bersama John Watrous.

Kamu akan mencipta pasangan terjalin antara dua Qubit dalam komputer kuantum dengan mewujudkan keadaan Bell $|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$. Menggunakan primitif Estimator, kamu boleh mendapatkan nilai jangkaan yang diperlukan ($\langle AB \rangle, \langle Ab \rangle, \langle aB \rangle$, dan $\langle ab \rangle$) secara terus untuk mengira nilai jangkaan dua kuantiti CHSH $\langle S_1\rangle$ dan $\langle S_2\rangle$. Sebelum diperkenalkannya primitif Estimator, kamu perlu membina nilai jangkaan daripada hasil ukuran.

Kamu akan mengukur Qubit kedua dalam asas $Z$ dan $X$. Qubit pertama juga akan diukur dalam asas ortogon, tetapi dengan sudut terhadap Qubit kedua, yang akan kita sapu antara $0$ dan $2\pi$. Seperti yang akan kamu lihat, primitif Estimator memudahkan pelaksanaan Circuit berparameter. Daripada mencipta siri Circuit CHSH, kamu hanya perlu mencipta *satu* Circuit CHSH dengan parameter yang menentukan sudut ukuran dan siri nilai fasa untuk parameter tersebut.

Akhirnya, kamu akan menganalisis hasil dan memplotnya berbanding sudut ukuran. Kamu akan nampak bahawa untuk julat sudut ukuran tertentu, nilai jangkaan kuantiti CHSH $|\langle S_1\rangle| > 2$ atau $|\langle S_2\rangle| > 2$, yang menunjukkan pelanggaran ketaksamaan CHSH.

In [2]:
# To run on hardware, select the backend with the fewest number of jobs in the queue
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
backend.name

'ibm_kingston'

### Create a parameterized CHSH circuit

First, we write the circuit with the parameter $\theta$, which we call `theta`. The [`Estimator` primitive](https://docs.quantum-computing.ibm.com/api/qiskit-ibm-runtime/qiskit_ibm_runtime.EstimatorV2) can enormously simplify circuit building and output analysis by directly providing expectation values of observables. Many problems of interest, especially for near-term applications on noisy systems, can be formulated in terms of expectation values. `Estimator` (V2) primitive can automatically change measurement basis based on the supplied observable.

In [3]:
theta = Parameter("$\\theta$")

chsh_circuit = QuantumCircuit(2)
chsh_circuit.h(0)
chsh_circuit.cx(0, 1)
chsh_circuit.ry(theta, 0)
chsh_circuit.draw(output="mpl", idle_wires=False, style="iqp")

<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/6c77e40a-0.avif" alt="Output of the previous code cell" />

### Cipta Circuit CHSH berparameter
Pertama, kita tulis Circuit dengan parameter $\theta$, yang kita panggil `theta`. Primitif [`Estimator`](https://docs.quantum-computing.ibm.com/api/qiskit-ibm-runtime/qiskit_ibm_runtime.EstimatorV2) boleh memudahkan pembinaan Circuit dan analisis output dengan besar dengan menyediakan nilai jangkaan boleh ukur secara terus. Banyak masalah yang menarik, terutamanya untuk aplikasi jangka pendek pada sistem bising, boleh diformulasikan dalam sebutan nilai jangkaan. Primitif `Estimator` (V2) boleh menukar asas ukuran secara automatik berdasarkan boleh ukur yang diberikan.

In [4]:
number_of_phases = 21
phases = np.linspace(0, 2 * np.pi, number_of_phases)
# Phases need to be expressed as list of lists in order to work
individual_phases = [[ph] for ph in phases]

![Output of the previous code cell](../docs/images/tutorials/chsh-inequality/extracted-outputs/6c77e40a-0.avif)

### Cipta senarai nilai fasa untuk ditetapkan kemudian
Selepas mencipta Circuit CHSH berparameter, kamu akan mencipta senarai nilai fasa untuk ditetapkan pada Circuit dalam langkah seterusnya. Kamu boleh menggunakan kod berikut untuk mencipta senarai 21 nilai fasa yang merentangi $0$ hingga $2 \pi$ dengan jarak yang sama, iaitu $0$, $0.1 \pi$, $0.2 \pi$, ..., $1.9 \pi$, $2 \pi$.

In [5]:
# <CHSH1> = <AB> - <Ab> + <aB> + <ab> -> <ZZ> - <ZX> + <XZ> + <XX>
observable1 = SparsePauliOp.from_list(
    [("ZZ", 1), ("ZX", -1), ("XZ", 1), ("XX", 1)]
)

# <CHSH2> = <AB> + <Ab> - <aB> + <ab> -> <ZZ> + <ZX> - <XZ> + <XX>
observable2 = SparsePauliOp.from_list(
    [("ZZ", 1), ("ZX", 1), ("XZ", -1), ("XX", 1)]
)

### Boleh Ukur
Sekarang kita perlukan boleh ukur untuk mengira nilai jangkaan. Dalam kes kita, kita melihat asas ortogon bagi setiap Qubit, membiarkan putaran $Y-$ berparameter bagi Qubit pertama menyapu asas ukuran hampir berterusan berbanding asas Qubit kedua. Oleh itu, kita akan memilih boleh ukur $ZZ$, $ZX$, $XZ$, dan $XX$.

In [6]:
target = backend.target
pm = generate_preset_pass_manager(target=target, optimization_level=3)

chsh_isa_circuit = pm.run(chsh_circuit)
chsh_isa_circuit.draw(output="mpl", idle_wires=False, style="iqp")

<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/9a5561eb-0.avif" alt="Output of the previous code cell" />

## Langkah 2: Optimumkan masalah untuk pelaksanaan perkakasan kuantum

Untuk mengurangkan jumlah masa pelaksanaan kerja, primitif V2 hanya menerima Circuit dan boleh ukur yang mematuhi arahan dan ketersambungan yang disokong oleh sistem sasaran (dirujuk sebagai Circuit dan boleh ukur ISA, iaitu instruction set architecture).

### Circuit ISA

In [7]:
isa_observable1 = observable1.apply_layout(layout=chsh_isa_circuit.layout)
isa_observable2 = observable2.apply_layout(layout=chsh_isa_circuit.layout)

![Output of the previous code cell](../docs/images/tutorials/chsh-inequality/extracted-outputs/9a5561eb-0.avif)

### Boleh Ukur ISA

Begitu juga, kita perlu mengubah boleh ukur supaya serasi dengan Backend sebelum menjalankan kerja dengan [`Runtime Estimator V2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run). Kita boleh melakukan transformasi menggunakan kaedah `apply_layout` pada objek `SparsePauliOp`.

In [8]:
# To run on a local simulator:
# Use the StatevectorEstimator from qiskit.primitives instead.

estimator = Estimator(mode=backend)

pub = (
    chsh_isa_circuit,  # ISA circuit
    [[isa_observable1], [isa_observable2]],  # ISA Observables
    individual_phases,  # Parameter values
)

job_result = estimator.run(pubs=[pub]).result()

## Langkah 3: Laksanakan menggunakan primitif Qiskit
Untuk melaksanakan keseluruhan eksperimen dalam satu panggilan kepada [`Estimator`](https://docs.quantum-computing.ibm.com/api/qiskit-ibm-runtime/qiskit_ibm_runtime.EstimatorV2).
Kita boleh mencipta primitif [Qiskit Runtime `Estimator`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) untuk mengira nilai jangkaan kita. Kaedah `EstimatorV2.run()` mengambil iterable bagi `primitive unified blocs (PUB)`. Setiap PUB adalah iterable dalam format `(circuit, observables, parameter_values: Optional, precision: Optional)`.

In [9]:
chsh1_est = job_result[0].data.evs[0]
chsh2_est = job_result[0].data.evs[1]

In [10]:
fig, ax = plt.subplots(figsize=(10, 6))

# results from hardware
ax.plot(phases / np.pi, chsh1_est, "o-", label="CHSH1", zorder=3)
ax.plot(phases / np.pi, chsh2_est, "o-", label="CHSH2", zorder=3)

# classical bound +-2
ax.axhline(y=2, color="0.9", linestyle="--")
ax.axhline(y=-2, color="0.9", linestyle="--")

# quantum bound, +-2√2
ax.axhline(y=np.sqrt(2) * 2, color="0.9", linestyle="-.")
ax.axhline(y=-np.sqrt(2) * 2, color="0.9", linestyle="-.")
ax.fill_between(phases / np.pi, 2, 2 * np.sqrt(2), color="0.6", alpha=0.7)
ax.fill_between(phases / np.pi, -2, -2 * np.sqrt(2), color="0.6", alpha=0.7)

# set x tick labels to the unit of pi
ax.xaxis.set_major_formatter(tck.FormatStrFormatter("%g $\\pi$"))
ax.xaxis.set_major_locator(tck.MultipleLocator(base=0.5))

# set labels, and legend
plt.xlabel("Theta")
plt.ylabel("CHSH witness")
plt.legend()
plt.show()

<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/f6267448-0.avif" alt="Output of the previous code cell" />

In the figure, the lines and gray areas delimit the bounds; the outer-most (dash-dotted) lines delimit the quantum-bounds ($\pm 2$), whereas the inner (dashed) lines delimit the classical bounds ($\pm 2\sqrt{2}$). You can see that there are regions where the CHSH witness quantities exceeds the classical bounds. Congratulations! You have successfully demonstrated the violation of CHSH inequality in a real quantum system!

## Tutorial survey

Please take this short survey to provide feedback on this tutorial. Your insights will help us improve our content offerings and user experience.

[Link to survey](https://your.feedback.ibm.com/jfe/form/SV_3xxAgm1SF1wGp9k)